# 7-1. ARIMA/GARCH/SV 점예측이 정말 4가지 차분설정 전부에서 같은가

`6-4`/`6-5`는 "점예측이 같아서 리샘플링 CRPS로 구별 안 됨"을 전제로 정규분포 CRPS를 썼다. 코드근거 + 실제 숫자로 검증한다.

- **근거**: `garch_mean_sigma`/`sv_mean_sigma`는 내부에서 `model.predict(n_periods=h)`만 호출 -- d,D와 무관하게 항상 같은 ARIMA 객체를 그대로 씀.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pmdarima as pm
from arch import arch_model
from scipy.optimize import minimize

H = 4
M_SEASON = 4
MIN_TRAIN = 20
TARGET_VAR = "X3_l1"

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", TARGET_VAR]].dropna()[TARGET_VAR].values.astype(float)
n = len(vals)
origins = list(range(MIN_TRAIN, n - H + 1))

CONFIGS = [(0, 0), (1, 0), (0, 1), (1, 1)]
CONFIG_LABELS = {(0, 0): "기본(무차분)", (1, 0): "1차차분", (0, 1): "계절차분", (1, 1): "둘다(1차+계절)"}
print(f"대표변수: {TARGET_VAR}, origin {len(origins)}개")


대표변수: X3_l1, origin 37개


## 2. `6-5`와 동일한 (평균,표준편차) 함수 그대로 사용

동기화 메모: 아래 세 함수는 `6-5`의 함수와 완전히 동일하다(검증 대상 코드를 그대로 가져와야 의미가 있어 의도적으로 복붙).


In [2]:
XI_MEAN, XI_VAR = -1.2704, np.pi ** 2 / 2
MAX_PHI = 0.97


def fit_arima_fixed(train, d, D, m=M_SEASON):
    seasonal = D > 0
    kwargs = dict(seasonal=seasonal, suppress_warnings=True, error_action="ignore", d=d)
    if seasonal:
        kwargs.update(m=m, D=D)
    return pm.auto_arima(train, **kwargs)


def arima_fc(model, h=H):
    return np.asarray(model.predict(n_periods=h))


def garch_fc(model, h=H):
    # sv_mean_sigma/garch_mean_sigma의 fc 계산부와 동일 -- model.predict()만 호출
    return np.asarray(model.predict(n_periods=h))


def _kalman_negloglik(params, z_centered):
    phi = MAX_PHI * np.tanh(params[0])
    sigma_eta2 = np.exp(params[1]) ** 2
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def sv_fc(model, h=H):
    # sv_mean_sigma의 fc 계산부와 동일 -- model.predict()만 호출(SV 고유 계산은 sigma_h에만 관여)
    return np.asarray(model.predict(n_periods=h))


## 3. 4가지 설정 x 5개 표본 origin에서 3개 모델의 fc 직접 비교

`fc`는 `model.predict(n_periods=h)`의 점예측 배열이다. `fc[0]`은 h=1 값만 대표로 표시(h=2~4도 동일 결과).

**비교 방향**: "같다"는 origin·차분설정을 고정하고 모델만 바꿨을 때의 비교다 -- 설정·origin이 다른 행끼리는 값이 다른 게 정상이고, 같은 행 안 세 모델 컬럼끼리만 항상 같다. 결론은 origin에 무관하므로 5개(처음/중간/끝 포함)만 확인한다.


In [3]:
sample_origins = [origins[0], origins[len(origins)//4], origins[len(origins)//2], origins[3*len(origins)//4], origins[-1]]
print("표본 origin:", sample_origins)

rows = []
for (d, D) in CONFIGS:
    cfg_label = CONFIG_LABELS[(d, D)]
    for origin in sample_origins:
        train = vals[:origin]
        model = fit_arima_fixed(train, d, D)
        fc_a = arima_fc(model)
        fc_g = garch_fc(model)
        fc_s = sv_fc(model)
        identical_ag = np.array_equal(fc_a, fc_g)
        identical_as = np.array_equal(fc_a, fc_s)
        rows.append({
            "설정": cfg_label, "origin": origin,
            "ARIMA fc[0]": round(fc_a[0], 6), "GARCH fc[0]": round(fc_g[0], 6), "SV fc[0]": round(fc_s[0], 6),
            "ARIMA==GARCH": identical_ag, "ARIMA==SV": identical_as,
        })

check_df = pd.DataFrame(rows)
display(check_df)


표본 origin: [20, 29, 38, 47, 56]


,설정,origin,ARIMA fc[0],GARCH fc[0],SV fc[0],ARIMA==GARCH,ARIMA==SV
0,기본(무차분),20,507.297934,507.297934,507.297934,True,True
1,기본(무차분),29,377.358063,377.358063,377.358063,True,True
2,기본(무차분),38,562.823567,562.823567,562.823567,True,True
3,기본(무차분),47,350.178279,350.178279,350.178279,True,True
4,기본(무차분),56,795.964528,795.964528,795.964528,True,True
5,1차차분,20,577.504321,577.504321,577.504321,True,True
6,1차차분,29,363.207537,363.207537,363.207537,True,True
7,1차차분,38,597.576789,597.576789,597.576789,True,True
8,1차차분,47,342.359871,342.359871,342.359871,True,True
9,1차차분,56,855.533532,855.533532,855.533532,True,True


In [4]:
all_identical = check_df["ARIMA==GARCH"].all() and check_df["ARIMA==SV"].all()
print(f"4가지 차분설정 x {len(sample_origins)}개 origin 전부에서 ARIMA==GARCH==SV fc 일치: {all_identical}")


4가지 차분설정 x 5개 origin 전부에서 ARIMA==GARCH==SV fc 일치: True


## 4. 결론

- **20개 조합(4설정 x 5origin) 전부에서 ARIMA/GARCH/SV 점예측이 부동소수점까지 완전히 동일.**
- 코드 구조상 필연: GARCH/SV 고유 계산(분산모델, 칼만필터)은 sigma_h에만 관여하고, fc는 셋 다 같은 ARIMA `predict()`를 재사용.
- `6-5`의 "정규분포 CRPS를 쓴다"는 전제는 4가지 차분설정 전부에서 유효.
